# Chunking Strategies for LLM and RAG Applications

This notebook explains **chunking** in a clear, practical way and then demonstrates different chunking strategies through well-commented Python code.

Chunking is one of the most important preprocessing steps in:

- Retrieval-Augmented Generation (RAG)
- Semantic search
- Document question-answering
- Vector databases
- Long-document summarisation
- LLM-based knowledge systems

The purpose of this notebook is not only to run code, but to help a reader understand **why chunking is required**, **how it is done**, and **what each line of code is doing**.

## 1. What is Chunking?

Large documents are difficult for an LLM or retrieval system to process as one single block. A 100-page report may contain many topics, sections, paragraphs, and examples. If the full report is converted into one embedding, the meaning becomes too broad and retrieval becomes weak.

**Chunking** means dividing a large text document into smaller meaningful parts called **chunks**.

Instead of this:

```text
Entire Document
       ↓
One huge embedding
```

we do this:

```text
Document
   ↓
Chunk 1
Chunk 2
Chunk 3
Chunk 4
   ↓
Embeddings
   ↓
Vector Database
```

When a user asks a question, the system retrieves only the most relevant chunks instead of the whole document.

## 2. Why is Chunking Necessary?

Suppose a 50-page report contains one small paragraph about forest fire prediction. If the entire report is stored as one unit, that paragraph may be hidden inside too much unrelated information.

Chunking improves:

1. **Retrieval accuracy** — relevant parts are easier to find.
2. **Context preservation** — related sentences remain together.
3. **LLM efficiency** — only useful text is passed to the model.
4. **Embedding quality** — each vector represents a focused idea.
5. **Token management** — chunks can be kept within model limits.

A good chunk should be neither too small nor too large.

- Too small: context is lost.
- Too large: meaning becomes diluted.

## 3. A Simple Example

Consider this short document:

```text
Artificial Intelligence is transforming healthcare.
Machine learning models can detect diseases from medical images.
Recent studies show improvement in diagnostic accuracy.
```

If we create sentence-level chunks, the result may be:

```text
Chunk 1: Artificial Intelligence is transforming healthcare.
Chunk 2: Machine learning models can detect diseases from medical images.
Chunk 3: Recent studies show improvement in diagnostic accuracy.
```

Now, if the user asks about disease detection, the second chunk can be retrieved directly.

## 4. Import Required Libraries

The code below uses mostly standard Python libraries. This makes the notebook easy to run in most environments.

We use:

- `re` for regular expression based splitting
- `dataclass` for storing chunking results cleanly
- `typing` for readable type hints
- `textwrap` for displaying long chunks neatly

In [ ]:
# Regular expressions help us split text using patterns.
# For example, we can split text wherever there is punctuation followed by a space.
import re

# dataclass allows us to create a simple class for storing results.
# It avoids writing repetitive class boilerplate code.
from dataclasses import dataclass

# These type hints make the code easier to understand.
# List[str] means a list of strings.
# Dict means a dictionary.
from typing import List, Dict, Callable

# textwrap is used only for neat printing of long chunks.
import textwrap

## 5. Sample Text Used for Demonstration

We will use one sample text containing headings, paragraphs, and sentences. This allows us to see how different chunking methods behave on the same document.

In [ ]:
sample_text = """
ARTIFICIAL INTELLIGENCE IN HEALTHCARE

INTRODUCTION

Artificial Intelligence is transforming healthcare by enabling faster diagnosis, personalised treatment, and improved hospital management. It allows computers to analyse large volumes of medical data and identify useful patterns.

Machine learning models are now used in medical imaging, pathology, genomics, and clinical decision support. These models can assist doctors, but they do not replace clinical judgement.

APPLICATIONS

In radiology, AI systems can examine X-rays, CT scans, and MRI images. They can highlight suspicious areas and help doctors prioritise urgent cases.

In public health, predictive models can estimate disease outbreaks and help authorities prepare resources in advance. This is especially useful in large populations where manual monitoring is difficult.

CHALLENGES

AI systems require high-quality data. If the training data is biased or incomplete, the output may also become unreliable.

Ethical issues such as privacy, consent, accountability, and transparency must be carefully addressed before AI systems are deployed in sensitive healthcare settings.
""".strip()

print(sample_text[:500])

## 6. Result Container

Each chunking method will return a `ChunkResult` object.

This object stores:

- Name of the method
- List of generated chunks
- Useful metadata such as number of chunks and chunk size

Using a common result format makes it easy to compare different strategies later.

In [ ]:
@dataclass
class ChunkResult:
    """
    Stores the output of a chunking method.

    Attributes
    ----------
    method:
        Name of the chunking strategy used.

    chunks:
        List of generated text chunks.

    metadata:
        Dictionary containing useful information such as number of chunks,
        chunk size, overlap, or total words.
    """
    method: str
    chunks: List[str]
    metadata: Dict

## 7. Helper Functions

These helper functions are used by different chunking strategies.

They keep the main chunking code clean and easy to understand.

In [ ]:
def split_into_sentences(text: str) -> List[str]:
    """
    Split text into sentences using a simple regular expression.

    The pattern looks for punctuation marks such as '.', '?', or '!'
    followed by whitespace. This is a lightweight alternative to using
    external NLP libraries such as NLTK or spaCy.
    """

    # Remove extra spaces from the beginning and end of text.
    text = text.strip()

    # If the input is empty, return an empty list.
    if not text:
        return []

    # Split after '.', '?', or '!' when followed by one or more spaces.
    sentences = re.split(r'(?<=[.!?])\s+', text)

    # Remove empty strings and extra spaces from each sentence.
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def split_into_paragraphs(text: str) -> List[str]:
    """
    Split text into paragraphs.

    A paragraph is usually separated by one or more blank lines.
    This function treats multiple newline characters as paragraph boundaries.
    """

    # Split text wherever there are one or more blank lines.
    paragraphs = re.split(r'\n\s*\n+', text.strip())

    # Clean each paragraph and remove empty results.
    return [paragraph.strip() for paragraph in paragraphs if paragraph.strip()]


def count_words(text: str) -> int:
    """
    Count words in a text using simple whitespace splitting.
    """
    return len(text.split())


def show_chunks(result: ChunkResult, max_width: int = 95) -> None:
    """
    Display chunks in a readable format.

    Parameters
    ----------
    result:
        ChunkResult returned by a chunking method.

    max_width:
        Width used for wrapping long lines on screen.
    """

    print(f"Method: {result.method}")
    print(f"Metadata: {result.metadata}")
    print("-" * max_width)

    for i, chunk in enumerate(result.chunks, start=1):
        print(f"\nCHUNK {i} | Words: {count_words(chunk)}")
        print(textwrap.fill(chunk, width=max_width))
        print("-" * max_width)

# Strategy 1: Fixed Size Chunking

## Concept

Fixed size chunking divides the document into chunks containing a fixed number of words.

Example:

```text
Chunk size = 30 words

Chunk 1 = Words 1-30
Chunk 2 = Words 31-60
Chunk 3 = Words 61-90
```

## Advantage

It is simple and fast.

## Limitation

It may cut a sentence or idea in the middle.

In [ ]:
def fixed_size_chunking(text: str, chunk_size: int = 50) -> ChunkResult:
    """
    Split text into fixed-size word chunks.

    Parameters
    ----------
    text:
        Input document.

    chunk_size:
        Number of words in each chunk.
    """

    # Step 1: Convert the full document into a list of words.
    # Example: "AI is useful" becomes ["AI", "is", "useful"]
    words = text.split()

    # This list will store the final chunks.
    chunks = []

    # Step 2: Move through the word list in jumps of 'chunk_size'.
    # If chunk_size = 50, i will be 0, 50, 100, 150, and so on.
    for i in range(0, len(words), chunk_size):

        # Step 3: Select words from position i up to i + chunk_size.
        chunk_words = words[i:i + chunk_size]

        # Step 4: Join selected words back into a normal text string.
        chunk_text = " ".join(chunk_words)

        # Step 5: Add this chunk to the final list.
        chunks.append(chunk_text)

    # Store useful information about the process.
    metadata = {
        "total_words": len(words),
        "chunk_size_words": chunk_size,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Fixed Size Chunking", chunks, metadata)


fixed_result = fixed_size_chunking(sample_text, chunk_size=50)
show_chunks(fixed_result)

# Strategy 2: Sliding Window Chunking

## Concept

Sliding window chunking is similar to fixed size chunking, but it keeps some overlap between consecutive chunks.

Example:

```text
Chunk size = 50 words
Overlap = 10 words
Step = 50 - 10 = 40 words

Chunk 1 = Words 1-50
Chunk 2 = Words 41-90
Chunk 3 = Words 81-130
```

The overlapping words help preserve context at boundaries.

## Why overlap matters

Without overlap, an important sentence may be split between two chunks. Overlap reduces this risk.

In [ ]:
def sliding_window_chunking(text: str, chunk_size: int = 60, overlap: int = 15) -> ChunkResult:
    """
    Split text into overlapping word chunks.

    Parameters
    ----------
    text:
        Input document.

    chunk_size:
        Number of words in each chunk.

    overlap:
        Number of words repeated from the previous chunk.
    """

    # Convert document into words.
    words = text.split()

    # Safety check: overlap must be smaller than chunk_size.
    # Otherwise, the loop may not move forward properly.
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    # Step tells us how far the window moves each time.
    # Example: chunk_size = 60 and overlap = 15, so step = 45.
    step = chunk_size - overlap

    chunks = []

    # Move through the document using the calculated step.
    for i in range(0, len(words), step):

        # Select a window of words.
        chunk_words = words[i:i + chunk_size]

        # Stop if no words are left.
        if not chunk_words:
            break

        # Join words into a chunk and store it.
        chunks.append(" ".join(chunk_words))

    metadata = {
        "total_words": len(words),
        "chunk_size_words": chunk_size,
        "overlap_words": overlap,
        "step_words": step,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Sliding Window Chunking", chunks, metadata)


sliding_result = sliding_window_chunking(sample_text, chunk_size=60, overlap=15)
show_chunks(sliding_result)

# Strategy 3: Sentence-Based Chunking

## Concept

Sentence-based chunking keeps complete sentences together.

Example:

```text
Sentences per chunk = 2

Chunk 1 = Sentence 1 + Sentence 2
Chunk 2 = Sentence 3 + Sentence 4
```

## Advantage

It avoids cutting sentences in the middle.

## Limitation

Some sentences may be very long, making chunk sizes uneven.

In [ ]:
def sentence_based_chunking(text: str, sentences_per_chunk: int = 3) -> ChunkResult:
    """
    Split text into chunks containing a fixed number of sentences.
    """

    # Step 1: Break the document into complete sentences.
    sentences = split_into_sentences(text)

    chunks = []

    # Step 2: Move through the sentence list in groups.
    for i in range(0, len(sentences), sentences_per_chunk):

        # Select a group of sentences.
        selected_sentences = sentences[i:i + sentences_per_chunk]

        # Join the selected sentences into one chunk.
        chunk_text = " ".join(selected_sentences)

        # Store the chunk.
        chunks.append(chunk_text)

    metadata = {
        "total_sentences": len(sentences),
        "sentences_per_chunk": sentences_per_chunk,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Sentence-Based Chunking", chunks, metadata)


sentence_result = sentence_based_chunking(sample_text, sentences_per_chunk=3)
show_chunks(sentence_result)

# Strategy 4: Paragraph-Based Chunking

## Concept

Paragraph-based chunking treats each paragraph as a chunk.

This is often natural because writers organise ideas into paragraphs.

## Advantage

It preserves the author's structure.

## Limitation

Paragraphs may vary widely in length.

In [ ]:
def paragraph_based_chunking(text: str) -> ChunkResult:
    """
    Split text into chunks based on paragraphs.
    """

    # Split text wherever there is a blank line.
    paragraphs = split_into_paragraphs(text)

    # In paragraph-based chunking, each paragraph becomes a chunk.
    chunks = paragraphs

    metadata = {
        "total_paragraphs": len(paragraphs),
        "num_chunks": len(chunks)
    }

    return ChunkResult("Paragraph-Based Chunking", chunks, metadata)


paragraph_result = paragraph_based_chunking(sample_text)
show_chunks(paragraph_result)

# Strategy 5: Character-Based Chunking

## Concept

Character-based chunking splits the document by number of characters.

Example:

```text
Chunk size = 300 characters

Chunk 1 = Characters 1-300
Chunk 2 = Characters 301-600
```

## Advantage

It is very simple and fast.

## Limitation

It may split words or sentences in unnatural places.

In [ ]:
def character_based_chunking(text: str, chunk_size: int = 300, overlap: int = 50) -> ChunkResult:
    """
    Split text into character-based chunks with optional overlap.
    """

    # Safety check for overlap.
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    # Calculate how far to move for each chunk.
    step = chunk_size - overlap

    chunks = []

    # Move across the text by character positions.
    for start in range(0, len(text), step):

        # Select characters from start to start + chunk_size.
        chunk = text[start:start + chunk_size]

        # Avoid adding empty chunks.
        if chunk.strip():
            chunks.append(chunk.strip())

    metadata = {
        "total_characters": len(text),
        "chunk_size_characters": chunk_size,
        "overlap_characters": overlap,
        "step_characters": step,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Character-Based Chunking", chunks, metadata)


character_result = character_based_chunking(sample_text, chunk_size=300, overlap=50)
show_chunks(character_result)

# Strategy 6: Token-Based Chunking

## Concept

LLMs do not read text exactly as humans read words. They process text in units called **tokens**.

A token may be:

- a full word
- part of a word
- punctuation
- a space-related symbol

For simplicity, this notebook demonstrates approximate token chunking using word-like tokens and punctuation.

In production systems, use the tokenizer of the actual model, for example `tiktoken` for OpenAI models.

In [ ]:
def simple_tokenize(text: str) -> List[str]:
    """
    A simple tokenizer for demonstration.

    This separates words and punctuation marks.
    It is not identical to an LLM tokenizer, but it is enough to explain the concept.
    """

    # \w+ captures words and numbers.
    # [^\w\s] captures punctuation marks.
    return re.findall(r"\w+|[^\w\s]", text)


def token_based_chunking(text: str, max_tokens: int = 80, overlap: int = 20) -> ChunkResult:
    """
    Split text into chunks based on approximate tokens.
    """

    tokens = simple_tokenize(text)

    if overlap >= max_tokens:
        raise ValueError("overlap must be smaller than max_tokens")

    step = max_tokens - overlap
    chunks = []

    for i in range(0, len(tokens), step):

        # Select a group of tokens.
        chunk_tokens = tokens[i:i + max_tokens]

        if not chunk_tokens:
            break

        # Join tokens with spaces for readability.
        # This is simple demonstration formatting.
        chunk_text = " ".join(chunk_tokens)

        # Clean spaces before punctuation for better display.
        chunk_text = re.sub(r"\s+([.,!?;:])", r"\1", chunk_text)

        chunks.append(chunk_text)

    metadata = {
        "total_tokens_approx": len(tokens),
        "max_tokens_per_chunk": max_tokens,
        "overlap_tokens": overlap,
        "step_tokens": step,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Token-Based Chunking", chunks, metadata)


token_result = token_based_chunking(sample_text, max_tokens=80, overlap=20)
show_chunks(token_result)

# Strategy 7: Hierarchical Chunking

## Concept

Hierarchical chunking respects the structure of a document.

It tries this order:

```text
Section → Paragraph → Sentence
```

If a section is small enough, it is kept as one chunk. If it is too large, it is broken into paragraphs. If a paragraph is still too large, it is broken into sentences.

## Why it is powerful

Research papers, reports, policy documents, manuals, and books usually have meaningful structure. Hierarchical chunking tries to preserve that structure.

In [ ]:
def hierarchical_chunking(text: str, max_words: int = 80) -> ChunkResult:
    """
    Split text using document structure.

    This function first detects headings written in uppercase.
    Then it attempts to keep sections together if they are not too large.
    Larger sections are split into paragraphs or sentences.
    """

    chunks = []

    # Step 1: Split text into paragraphs.
    paragraphs = split_into_paragraphs(text)

    # Step 2: Group paragraphs into sections.
    # A simple assumption is used here:
    # paragraphs written mostly in uppercase are treated as headings.
    sections = []
    current_section = []

    for paragraph in paragraphs:

        # Check whether paragraph looks like a heading.
        is_heading = paragraph.isupper() and len(paragraph.split()) <= 8

        if is_heading:
            # If a new heading starts and we already have content,
            # save the previous section first.
            if current_section:
                sections.append("\n\n".join(current_section))
                current_section = []

        # Add heading or paragraph to current section.
        current_section.append(paragraph)

    # Add the last section after loop ends.
    if current_section:
        sections.append("\n\n".join(current_section))

    # Step 3: Process each section.
    for section in sections:

        # If section is small enough, keep it as one chunk.
        if count_words(section) <= max_words:
            chunks.append(section)

        else:
            # If section is large, split it into paragraphs.
            section_paragraphs = split_into_paragraphs(section)

            for paragraph in section_paragraphs:

                # Keep paragraph if it is within limit.
                if count_words(paragraph) <= max_words:
                    chunks.append(paragraph)

                else:
                    # If paragraph is also large, split into sentences.
                    sentences = split_into_sentences(paragraph)

                    temp_chunk = []
                    temp_word_count = 0

                    for sentence in sentences:
                        sentence_words = count_words(sentence)

                        # If adding this sentence would exceed limit,
                        # save the existing temporary chunk first.
                        if temp_chunk and temp_word_count + sentence_words > max_words:
                            chunks.append(" ".join(temp_chunk))
                            temp_chunk = []
                            temp_word_count = 0

                        temp_chunk.append(sentence)
                        temp_word_count += sentence_words

                    # Save remaining sentences.
                    if temp_chunk:
                        chunks.append(" ".join(temp_chunk))

    metadata = {
        "max_words_per_chunk": max_words,
        "detected_sections": len(sections),
        "num_chunks": len(chunks)
    }

    return ChunkResult("Hierarchical Chunking", chunks, metadata)


hierarchical_result = hierarchical_chunking(sample_text, max_words=80)
show_chunks(hierarchical_result)

# Strategy 8: Recursive Chunking

## Concept

Recursive chunking attempts to split text using the most meaningful separator first.

A typical order is:

```text
Paragraph break → Sentence boundary → Word boundary
```

The idea is simple:

1. Try to keep paragraphs intact.
2. If a paragraph is too large, split into sentences.
3. If a sentence is too large, split by words.

This is conceptually similar to recursive text splitters used in many RAG frameworks.

In [ ]:
def recursive_chunking(text: str, max_words: int = 80) -> ChunkResult:
    """
    Recursively split text while preserving meaning as much as possible.
    """

    def split_recursively(piece: str) -> List[str]:
        """
        Internal helper function that splits one piece of text.
        It calls itself again if the piece is still too large.
        """

        # Base case: if the piece is already small enough, return it directly.
        if count_words(piece) <= max_words:
            return [piece.strip()]

        # First attempt: split by paragraphs.
        paragraphs = split_into_paragraphs(piece)
        if len(paragraphs) > 1:
            result = []
            for paragraph in paragraphs:
                result.extend(split_recursively(paragraph))
            return result

        # Second attempt: split by sentences.
        sentences = split_into_sentences(piece)
        if len(sentences) > 1:
            result = []
            current_chunk = []
            current_words = 0

            for sentence in sentences:
                sentence_word_count = count_words(sentence)

                if current_chunk and current_words + sentence_word_count > max_words:
                    result.append(" ".join(current_chunk))
                    current_chunk = []
                    current_words = 0

                current_chunk.append(sentence)
                current_words += sentence_word_count

            if current_chunk:
                result.append(" ".join(current_chunk))

            return result

        # Final attempt: if even one sentence is too long, split by words.
        words = piece.split()
        return [" ".join(words[i:i + max_words]) for i in range(0, len(words), max_words)]

    chunks = split_recursively(text)

    metadata = {
        "max_words_per_chunk": max_words,
        "num_chunks": len(chunks)
    }

    return ChunkResult("Recursive Chunking", chunks, metadata)


recursive_result = recursive_chunking(sample_text, max_words=80)
show_chunks(recursive_result)

# 9. Comparing All Chunking Strategies

Now we run all strategies on the same sample text and compare their outputs.

This comparison helps us understand that different strategies produce different numbers and sizes of chunks.

In [ ]:
# Store all chunking functions in one list.
# Each entry contains a readable name and a function call.
strategies: Dict[str, Callable[[], ChunkResult]] = {
    "Fixed Size": lambda: fixed_size_chunking(sample_text, chunk_size=50),
    "Sliding Window": lambda: sliding_window_chunking(sample_text, chunk_size=60, overlap=15),
    "Sentence Based": lambda: sentence_based_chunking(sample_text, sentences_per_chunk=3),
    "Paragraph Based": lambda: paragraph_based_chunking(sample_text),
    "Character Based": lambda: character_based_chunking(sample_text, chunk_size=300, overlap=50),
    "Token Based": lambda: token_based_chunking(sample_text, max_tokens=80, overlap=20),
    "Hierarchical": lambda: hierarchical_chunking(sample_text, max_words=80),
    "Recursive": lambda: recursive_chunking(sample_text, max_words=80),
}

# Build comparison table as a list of dictionaries.
comparison = []

for name, function in strategies.items():
    result = function()
    word_counts = [count_words(chunk) for chunk in result.chunks]

    comparison.append({
        "Strategy": name,
        "Number of Chunks": len(result.chunks),
        "Minimum Words": min(word_counts),
        "Maximum Words": max(word_counts),
        "Average Words": round(sum(word_counts) / len(word_counts), 2)
    })

# Display comparison in a simple readable form.
for row in comparison:
    print(row)

# 10. Practical Guidance: Which Chunking Strategy Should You Use?

| Strategy | Best For | Strength | Weakness |
|---|---|---|---|
| Fixed Size | Simple experiments | Fast and easy | May break meaning |
| Sliding Window | General RAG | Preserves boundary context | Creates duplicate text |
| Sentence Based | Question answering | Keeps sentences intact | Uneven chunk sizes |
| Paragraph Based | Articles and reports | Preserves writing structure | Long paragraphs may be too large |
| Character Based | Quick preprocessing | Very fast | Can split words unnaturally |
| Token Based | LLM applications | Matches model limits better | Needs model-specific tokenizer for accuracy |
| Hierarchical | Research papers, reports, manuals | Preserves document structure | More complex logic |
| Recursive | Production RAG systems | Good balance of structure and size | Requires careful parameter tuning |

For most RAG systems, good starting options are:

1. Recursive chunking
2. Sliding window chunking
3. Token-based chunking
4. Hierarchical chunking

# 11. Final Takeaway

Chunking is not merely a technical operation. It directly affects the quality of retrieval and the quality of answers generated by an LLM.

A good chunking strategy should preserve meaning, maintain context, and stay within practical size limits.

There is no single best method for every document. The best strategy depends on:

- Type of document
- Length of document
- Structure of document
- Retrieval requirement
- LLM context window
- Embedding model used

In real-world applications, the chunking strategy should always be tested with actual user queries and retrieval results.